In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import sys

class _DL:
    def __init__(self, f):
        self.t=sys.stdout; self.l=open(f, 'w')
    def write(self, m):
        self.t.write(m); self.l.write(m); self.l.flush()
    def flush(self):
        self.t.flush(); self.l.flush()
sys.stdout = _DL('live_output_part7.log')

# ======================================================================
# QUANTIZATION OPS (STE Base)
# ======================================================================
def quantize_w4(w):
    scale = 7.0 / w.abs().max().clamp(min=1e-8)
    w_q = torch.round(w * scale).clamp(-8, 7) / scale
    return w + (w_q - w).detach()

def compute_projection_matrix(A, k):
    if A.shape[0] > 1000:
        _, _, V = torch.svd_lowrank(A, q=k)
        U_k = V
    else:
        L, V = torch.linalg.eigh(A)
        idx = torch.argsort(L, descending=True)
        V = V[:, idx]
        U_k = V[:, :k]
    return U_k @ U_k.T

class ToyMatrixModel(nn.Module):
    def __init__(self, d_in, d_hidden, quant_type='W4A8'):
        super().__init__()
        self.d_hidden = d_hidden
        self.quant_type = quant_type
        self.W_in = nn.Parameter(torch.randn(d_in, d_hidden) / np.sqrt(d_in))
        self.W_out = nn.Parameter(torch.randn(d_hidden, d_in) / np.sqrt(d_hidden))
        
    def _quantize(self, w):
        if self.quant_type == 'W4A8': return quantize_w4(w)
        return w
        
    def forward(self, x):
        W_in_sim = self._quantize(self.W_in)
        W_out_sim = self._quantize(self.W_out)
        h = F.relu(x @ W_in_sim)
        return h @ W_out_sim

# ======================================================================
# EXPERIMENT 4: Targeted FPE Escape Verification (Part 7)
# Pure Deng et al. Spectral Gap / Alignment Superposition metrics
# ======================================================================
def run_deng_superposition_experiment(device):
    print("\n--- EXPERIMENT 4 (PART 7): TARGETED NEURON EXPANSION VIA DENG ALIGNMENT TRAP ---")
    d_in, d_hidden = 1024, 256
    target_k = 100 
    n_steps = 15000
    
    model = ToyMatrixModel(d_in, d_hidden).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.15, momentum=0.9)
    
    # Random structured dataset to create feature competition (superposition setup)
    X = torch.randn(2048, d_in, device=device)
    Y = X @ (torch.randn(d_in, d_in, device=device) * 0.5)
    
    # Deng Metric Traps
    # E_AL_t is the expected spectral alignment under random Gaussian orthogonal condition
    e_al_target = target_k / d_hidden  
    trap_threshold = e_al_target * 1.5  # We are in superposition if alignment significantly exceeds basic capacity
    
    print(f"Tracking Network Geometry: d_hidden = {d_hidden}, Feature Rank = {target_k}")
    print(f"Expected Minimum Alignment (m/d): {e_al_target:.3f}")
    print(f"Triggering expansion when Empirical AL_t > {trap_threshold:.3f} (Superposition Limit)")
    
    expansions = 0
    
    for step in range(n_steps):
        idx = torch.randint(0, 2048, (1024,))
        x_b, y_b = X[idx], Y[idx]
        
        y_hat = model(x_b)
        loss = F.mse_loss(y_hat, y_b)
        
        optimizer.zero_grad()
        loss.backward()
        
        # Deng Metric: Spectral Alignment Computation
        g_mat = model.W_in.grad.detach() 
        A = g_mat @ g_mat.T 
        P = compute_projection_matrix(A, target_k)
        
        num = torch.norm(P @ g_mat)**2
        den = torch.norm(g_mat)**2 + 1e-12
        al_t = (num / den).item()

        if step % 100 == 0:
            print(f"  [Step {step:4d}] Loss: {loss.item():.4f} | AL_t: {al_t:.3f} | E_AL: {e_al_target:.3f} | Trap Threshold: {trap_threshold:.3f}")

        # Deng Trap Detection (allow multiple expansions but with a cooldown)
        if al_t > trap_threshold and (step - (fpe_step if fpe_step else -500)) > 500:
            fpe_step = step
            expansions += 1
            print(f"\n[Step {step}] 🎯 DENG SUPERPOSITION TRAP DETECTED:")
            print(f"  - Empirical Alignment (AL_t) = {al_t:.3f} EXCEEDS Threshold = {trap_threshold:.3f}")
            print(f"  --> The {model.d_hidden} neurons are stuck mapping {target_k} orthogonal gradient features.")
            print(f"  --> Executing TARGETED Fractional Expansion to {model.d_hidden + 128} neurons to relieve capacity constraint!")
            
            old_w_in = model.W_in.detach()
            old_w_out = model.W_out.detach()
            
            spawn_w_in = torch.randn(d_in, 128, device=device) * 0.01
            spawn_w_out = torch.randn(128, d_in, device=device) * 0.01
            
            model.W_in = nn.Parameter(torch.cat([old_w_in, spawn_w_in], dim=1))
            model.W_out = nn.Parameter(torch.cat([old_w_out, spawn_w_out], dim=0))
            model.d_hidden += 128
            
            # Recalculate Deng E_AL after expansion
            e_al_target = target_k / model.d_hidden
            trap_threshold = e_al_target * 1.5
            print(f"  --> Network Capacity Re-Evaluated: New (m/d) = {e_al_target:.3f} | New Threshold = {trap_threshold:.3f}")
            
            # Decay learning rate drastically to stabilize new parameter state
            optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
            
        optimizer.step()
        
    if expansions > 0:
        print(f"\n✅ SUCCESS: Network gracefully expanded out of the Deng Superposition Trap {expansions} separate times via structural detonation.")
    else:
        print("\n❌ FAILURE: Network never reached a strong Deng Superposition alignment state. Increase problem rank or decrease model dimension.")

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run_deng_superposition_experiment(device)

if __name__ == '__main__':
    main()
